### 1. 준비하기 - 모듈 불러오기

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  
MODEL = "gpt-5.4-mini"           # 상수/변수는 보통 대문자로 한다

### Phase 1: 아키텍처 및 데이터 설계
LLM 혼자 모든 정보를 알 수 없으므로(할루시네이션 방지), 외부 도구와 데이터를 연동하는 구조를 짜야 합니다.

에이전트 프레임워크 선정: LangGraph나 CrewAI를 추천합니다. (복잡한 일정을 조율하고 상태를 유지하는 데 유리합니다.)

외부 API 연동:

장소 정보: 한국관광공사 TourAPI (전국 관광지, 식당, 숙박 정보 제공)

비용 계산: 유가 정보 API(OPINET), 대중교통 요금 정보, 숙박 예약 사이트 크롤링 등

지도/경로: Naver Map 또는 Kakao Map API (장소 간 이동 거리 및 소요 시간 계산)

### Phase 2: 멀티 에이전트 역할 분담 (Multi-Agent System)
하나의 프롬프트로 처리하기보다 기능을 쪼개어 전문 에이전트들을 배치합니다.

Planner Agent: 사용자의 지역, 기간, 테마(힐링, 액티비티 등)를 분석해 대략적인 동선을 짭니다.

Finance Agent: 선택된 경로의 교통비, 입장료, 예상 식비 등을 실시간 API나 DB를 통해 계산합니다.

Taste Matcher Agent: 사용자의 기존 취향(예: "북적이는 곳 싫어함", "전통 시장 선호")을 기반으로 추천된 후보지들에 점수를 매깁니다.

### Phase 3: 단계별 인터랙션 로직 구현
에이전트가 일방적으로 결과를 내놓는 것이 아니라, **'협업'**하는 느낌을 주어야 합니다.

Step 1. 정보 수집: LLM이 사용자에게 필요한 정보(누구와 가는지, 못 먹는 음식은 무엇인지 등)를 질문하여 '사용자 프로필'을 생성합니다.

Step 2. 초안 생성: 인기순/테마별로 2~3가지의 대안 경로와 예상 총비용을 제안합니다.

Step 3. 피드백 및 수정: "2일 차 점심은 좀 더 가벼운 걸 먹고 싶어" 같은 요청을 받으면 Finance Agent와 Planner가 연동되어 실시간으로 비용과 일정을 업데이트합니다.

### Phase 4: 비용 산출 및 최적화 엔진
이 부분이 프로젝트의 차별점이 될 수 있습니다.

Deterministic Logic 도입: LLM은 계산에 약하므로, 구체적인 수치는 Python 코드(Code Interpreter 방식)가 수행하도록 합니다.

예: (이동 거리 * 유가) + 숙박비 + (식비 * 인원수) = 예상 비용

지리적 최적화: 장소들 사이의 거리를 계산하여 동선이 꼬이지 않게 재정렬하는 로직을 추가합니다.

### Phase 5: 개인화 메모리 시스템 (Long-term Memory)
사용자가 다음에 다시 접속했을 때 "지난번 경주 여행처럼 조용한 곳 위주로 짜줘"라고 하면 기억할 수 있어야 합니다.

Vector DB(예: Pinecone, Chroma): 사용자의 과거 대화 기록과 선택했던 장소들을 임베딩하여 저장하고, 추천 시 이를 검색(RAG)하여 반영합니다.

추천 기술 스택 예시
언어: Python

LLM: GPT-4o (추론용) 또는 Claude 3.5 Sonnet (창의적 일정 작성)

프레임워크: LangChain + LangGraph

패키지 관리: uv (제안해주신 대로 빠르고 효율적인 관리를 위해 적극 권장합니다.)

데이터베이스: PostgreSQL (사용자 데이터) + Vector DB (취향 데이터)

### 📚 국내 여행 추천 LLM 에이전트 구현 완료!

#### ✅ 구현된 기능

1. **사용자 입력 수집**
   - 지역 선택 (7개 지역)
   - 여행 기간 설정 (1-7일)
   - 인원수 입력 (1-10명)
   - 테마 선택 (10가지)

2. **LLM 기반 추천 생성**
   - GPT-4o-mini를 활용한 맞춤형 추천
   - JSON 형식 구조화된 응답
   - 폴백 메커니즘 내장

3. **지능형 비용 계산**
   - 교통비 (지역별 상이)
   - 식사비 (일별)
   - 숙박비 (박수별)
   - **단체할인 자동 적용** (최대 20%)

4. **사용자 선택 지원**
   - 3가지 추천 코스 제시
   - 실시간 비용 계산
   - 최종 계획 수립

#### 🏗️ LangGraph 워크플로우

```
입력 수집 → LLM 추천 생성 → 추천 결과 표시
    ↓
비용 계산 → 비용 표시 → 코스 선택 → 최종 요약
```

#### 📊 기능 설명

## 아키텍처 및 데이터 설계

LLM이 모든 정보를 알 수 없으므로(할루시네이션 방지), **외부 데이터**와 **구조화된 계산**을 연동합니다.

**에이전트 프레임워크**: LangGraph
- 상태 관리 (State)
- 단계적 실행 (Workflow)
- 조건부 라우팅 가능

**데이터 구조**:


In [3]:
question = """
여행 코스 추천해주는 llm 에이전트를 만드는 프로젝트를 할건데, 
경로 추천뿐만 아니라 예상 비용을 계산해주고, 
추천 후보들 중에 선택하는데까지도 도움을 줄 수 있는 에이전트를 만들어보고 싶어. 
어떻게 단계별로 프로젝트를 수행할 수 있을까?
"""

response = client.responses.create(
    model=MODEL,
    reasoning={"effort" : "none"},
    input=question
)
print(response.output_text)

좋아요. 이 프로젝트는 단순한 “여행지 추천 챗봇”이 아니라, **경로 설계 + 비용 추정 + 후보 비교 + 의사결정 지원**까지 하는 **여행 플래닝 에이전트**로 설계하면 꽤 강력한 결과물을 만들 수 있습니다.  
아래처럼 **단계별로 쪼개서** 진행하면 구현 난이도와 검증이 훨씬 쉬워집니다.

---

## 1) 목표를 먼저 명확히 정의하기
먼저 에이전트가 무엇을 해줘야 하는지 범위를 정합니다.

### 핵심 기능
1. **여행 조건 수집**
   - 출발지, 목적지, 여행 기간
   - 인원수, 예산, 여행 스타일
   - 선호(맛집, 자연, 쇼핑, 박물관, 휴양 등)
   - 이동수단 선호(도보/대중교통/렌터카 등)

2. **경로 추천**
   - 일정별 추천 루트 생성
   - 장소 간 이동 동선 고려
   - 시간대별 적절한 방문지 배치

3. **예상 비용 계산**
   - 교통비
   - 숙박비
   - 식비
   - 입장료/체험비
   - 예비비

4. **후보 비교**
   - A안, B안, C안처럼 여러 일정 제시
   - 총비용, 이동량, 만족도, 여유도 비교

5. **선택 도움**
   - 사용자의 선호/제약 조건에 맞춰 추천
   - “이 일정은 가성비가 좋음”, “이 일정은 이동이 적어 피로도가 낮음” 같은 설명

---

## 2) 문제를 작은 하위 문제로 분해하기
이 프로젝트는 한 번에 만들기보다 아래 모듈로 나누는 것이 좋습니다.

### 모듈 A. 사용자 요구사항 파싱
입력:
- 자연어로 된 여행 요청

출력:
- 구조화된 여행 조건 JSON

예:
```json
{
  "destination": "일본 오사카",
  "days": 3,
  "travelers": 2,
  "budget": 1000000,
  "preferences": ["맛집", "쇼핑", "야경"],
  "constraints": ["걷는 시간 최소화"]
}
```

---

### 모듈 B. 장소 후보 수집
여행지별 POI(Point of Inter

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="너가 아까 추천해준 메뉴가 뭐더라?",
    previous_response_id=response.id,
    max_output_tokens=200
)
print(response.output_text)